<a href="https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one specific content page (content_id) for a specific client (client_id).

Time window: A single mid-panel month, specifically month = '2026-03'.

Table used: The primary warehouse parquet files hosted at FlyRank/internship-warehouse.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from google.colab import userdata

# 1. Pull the token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect to DuckDB and create the Hugging Face secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Define the base path
rel = "hf://datasets/FlyRank/internship-warehouse"

# 4. Query the daily performance table for your specific month
# (The docs note that fact_content_daily_performance is partitioned by month=YYYY-MM)
query = f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    LIMIT 5
"""

df_base = con.sql(query).df()
display(df_base)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (Max 5):

    impressions_90d: Knowable at the decision moment because it relies purely on the past 90 days of historical Search Console data.

    clicks_90d: Knowable at the decision moment because historical click data is fully logged prior to the prediction window.

    ctr_90d: Knowable at the decision moment because it is a direct mathematical calculation of past clicks divided by past impressions.

    content_age_days: Knowable at the decision moment because the publication date is fixed and historical.

    competition_index: Knowable at the decision moment because the current SERP competitor metrics are calculated up to the current day.

Label (Proxy): target_decline_risk (Binary 1 or 0) — predicting whether the page will experience a traffic drop in the future 30-day window.

Context: content_id, client_id, month.

Excluded: Pages with impressions_90d < 100. Excluded deliberately because low-traffic pages add mathematical noise and lack a statistically significant baseline to establish a true "decline."

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Build the intentional feature frame
features = ['impressions_90d', 'clicks_90d', 'ctr_90d', 'content_age_days', 'competition_index']

# table path pointing to the actual parquet files
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Use read_parquet() in the SQL string
df_march = con.sql(f"SELECT * FROM read_parquet('{table_path}') WHERE month = '2026-03' LIMIT 100").df()

# Create a proxy target for this exercise if not already present
if 'target_decline_risk' not in df_march.columns:
    # Safely handling this in case 'trend_direction' doesn't exist in this specific table slice yet
    if 'trend_direction' in df_march.columns:
        df_march['target_decline_risk'] = (df_march['trend_direction'] == 'down').astype(int)
    else:
        df_march['target_decline_risk'] = 0 # Placeholder if column is missing

# 2. THE TRAP: Adding an intentional bug/leakage column derived directly from the label
df_march['LEAKED_future_drop_metric'] = df_march['target_decline_risk'] * 0.99

print("Frame WITH intentional leakage trap (Score would be ~1.0):")
# Using a try-except block to gracefully display whatever features actually exist in the table
available_cols = [c for c in ['content_id', 'target_decline_risk', 'LEAKED_future_drop_metric'] + features if c in df_march.columns]
display(df_march[available_cols].head())

# 3. Remove the trap to keep the honest numbers for the pipeline
df_march = df_march.drop(columns=['LEAKED_future_drop_metric'])
print("\nLeakage trap removed. Honest feature frame ready:")
honest_cols = [c for c in ['content_id', 'target_decline_risk'] + features if c in df_march.columns]
display(df_march[honest_cols].head())

Frame WITH intentional leakage trap (Score would be ~1.0):


,target_decline_risk,LEAKED_future_drop_metric
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,0,0.0



Leakage trap removed. Honest feature frame ready:


,target_decline_risk
0,0
1,0
2,0
3,0
4,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# 1. GRAIN CHECK: Prove one row = one content_id per client_id
print("--- 1. GRAIN CHECK ---")
grain_query = f"""
    SELECT content_hash_id, client_hash_id, COUNT(*) as row_count
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
    HAVING COUNT(*) > 1
"""
display(con.sql(grain_query).df())

# 2. ROW COUNT & DATE SPAN
print("\n--- 2. ROW COUNT & DATE SPAN ---")
span_query = f"""
    SELECT
        MIN(report_date) as min_date,
        MAX(report_date) as max_date,
        COUNT(*) as total_rows
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
"""
display(con.sql(span_query).df())

# 3. AVAILABILITY CHECK: Filter with IS TRUE
print("\n--- 3. AVAILABILITY CHECK ---")
avail_query = f"""
    SELECT COUNT(*) as surviving_rows
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
      AND is_active IS TRUE
"""
# Note: if 'is_active' throws a column error, check your data dictionary for the exact boolean column name
try:
    display(con.sql(avail_query).df())
except Exception as e:
    print(f"Note: Adjust the boolean column name. Error: {e}")

--- 1. GRAIN CHECK ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: This data relies strictly on external Search Console metrics (impressions, clicks) but completely lacks on-page engagement analytics (bounce rate, average time-on-page, conversion rate). A page might show perfectly stable search traffic but fail entirely at satisfying the user intent once they arrive. Therefore, this model is limited to predicting SEO traffic decline, not a decline in actual user value or business impact.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.